In [ ]:
# Imports and Environment Setup 

import os
import time
from dotenv import load_dotenv

# Import LangChain components
from langchain_community.embeddings import OllamaEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

# Load the API key from the .env file
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

# Check if the API key is available
if not api_key:
    raise ValueError("Google API Key not found. Please set it in the .env file.")
else:
    print("Libraries imported and API key loaded successfully.")

Libraries imported and API key loaded successfully.


In [13]:
# Constants and Custom Prompt Template

# Define paths for documents and vector store
PDF_DOCS_PATH = "./documents"
CHROMA_PERSIST_PATH = "./db/chroma_store"

# prompt for the LLM's behavior.
custom_prompt_template = """
You are a helpful and honest medical information assistant.
Your task is to provide answers based on the provided context.
You can also provide answer based on your knowledge
Your answers should be clear and concise.
Do not mention that you are getting the information from a provided text.

Context: {context}
Chat History: {chat_history}

Question: {question}
Helpful Answer:
"""

def create_custom_prompt():
    """Creates a custom prompt template for the RAG chain."""
    return PromptTemplate(
        template=custom_prompt_template,
        input_variables=["context", "chat_history", "question"]
    )

print("Constants and prompt template function defined.")

Constants and prompt template function defined.


In [14]:
# Vector Store Function(with Ollama)

def get_vectorstore():
    """
    Processes PDF documents using a local Ollama model for embeddings.
    Creates or loads a persistent Chroma vector store.
    """
    # Using Ollama model for creating embeddings
    embeddings = OllamaEmbeddings(model="llama3.2:1b")

    if os.path.exists(CHROMA_PERSIST_PATH):
        # Load the existing vector store
        print("Loading existing vector store...")
        vector_store = Chroma(persist_directory=CHROMA_PERSIST_PATH, embedding_function=embeddings)
    else:
        # Create a new vector store
        print("Creating new vector store...")
        
        pdf_files = [f for f in os.listdir(PDF_DOCS_PATH) if f.endswith(".pdf")]
        if not pdf_files:
            raise FileNotFoundError(f"No PDF files found in '{PDF_DOCS_PATH}'.")

        documents = []
        for pdf_file in pdf_files:
            loader = PyPDFLoader(os.path.join(PDF_DOCS_PATH, pdf_file))
            documents.extend(loader.load())
        
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
        chunks = text_splitter.split_documents(documents)
        
        # Create and persist the vector store using the loaded chunks
        vector_store = Chroma.from_documents(chunks, embeddings, persist_directory=CHROMA_PERSIST_PATH)
        print("Vector store created successfully.")
    
    return vector_store

print("Vector store function with Ollama embeddings defined.")

Vector store function with Ollama embeddings defined.


In [15]:
# Conversational Chain Function(with Gemini)

def get_conversational_rag_chain(vector_store):
    """
    Creates the main conversational retrieval chain using Google Gemini for generation.
    """
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash-latest", google_api_key=api_key, temperature=0.3)

    # Set up memory to store chat history
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True
    )

    # Create the retriever from the vector store
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    # Create the custom prompt
    prompt = create_custom_prompt()

    # Create the conversational chain
    chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=retriever,
        memory=memory,
        combine_docs_chain_kwargs={"prompt": prompt},
        verbose=False
    )
    
    return chain

print("Conversational RAG chain function with Gemini model defined.")

Conversational RAG chain function with Gemini model defined.


In [ ]:
vector_store = get_vectorstore()
qa_chain = get_conversational_rag_chain(vector_store)
print("embedding done")

C:\Users\akmal\AppData\Local\Temp\ipykernel_21660\3109348910.py:9: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="llama3.2:1b")


Creating new vector store...
Vector store created successfully.
✅ Chatbot is ready! You can now ask questions in the next cell.


C:\Users\akmal\AppData\Local\Temp\ipykernel_21660\371390441.py:11: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


In [16]:
# Interactive Chat

# Initialize chat history once.
try:
    chat_history
except NameError:
    chat_history = []

# --- Ask qn ---
question = "what disease was i previously asked"
# --------------------------

if question.lower() == 'exit':
    print("Chat ended.")
else:
    # Get the result from the chain
    result = qa_chain({"question": question, "chat_history": chat_history})
    
    # Append the current question and the answer to the history
    chat_history.append((question, result["answer"]))
    
    # Print the answer
    print("AI:", result["answer"])

AI: Previously, you asked about the symptoms of fever (including influenza and dengue fever), pneumonia, how to stay healthy, and what cancer is.
